In [1]:
from __future__ import annotations

import uuid
from enum import Enum
import errno
import sys
import time
from pathlib import Path
from typing import Iterable, Iterator, Optional
import logging
import os
import re
from importlib import reload

from debugpy.common.log import LEVELS

In [2]:
# import my packages

from identifier import Id, CompositeId
import record_pb2
from meta import Run, Dir, File
from db import Db

In [3]:
# open database for reading

db_name = 'test-db-for-db'
path = Path(db_name)

readonly = True
# readonly = False

create=False
# create=True

the_db = Db.open(path, readonly=readonly, create=create)
env = the_db.env

In [4]:
# import identifier
# reload(identifier)
# from identifier import Id, CompositeId

In [5]:
# import db
# reload(db)
# from db import Db

In [ ]:
# show lmdb env info

env.flags(), env.path(), env.info()

In [43]:
env.close()

In [58]:
# list all key-value pairs in the db, using lmdb's env

with env.begin(write=False) as txn:
    num_listed = 0
    with txn.cursor() as cur:
        for key, value in cur.iternext():
            # if key.startswith(b'r:'):
            #     print(key, value)
            print(f'key={key}', f'value={value}')
            num_listed += 1
    print(f'num_listed={num_listed}')

key=b'd:\x00*\x00\x01' value=b'\x08\x01\x12\x04\x00*\x00\x01"\x01.-f\xe6@F2\x101111222233334444:\x105555666677778888'
key=b'd:\x00*\x00\x02' value=b'\x08\x01\x10*\x18\x02*\x01.5f\xe6@F'
key=b'dhd:' value=b'\x00*\x00\x02'
key=b'dhd:UUffww\x88\x88\x11\x11""33DD' value=b'\x00*\x00\x01'
key=b'dhf:' value=b'\x00*\x00\x02'
key=b'dhf:\x11\x11""33DDUUffww\x88\x88' value=b'\x00*\x00\x01'
key=b'f:\x00*\x00\x01\x00\x01' value=b'\x08\x01\x1a\x0csomothername(\x01\x10*\x18\x018\xb2\xf2\x19'
key=b'f:\x00*\x00\x01\x00\x02' value=b'\x08\x01\x12\x06\x00*\x00\x01\x00\x02\x1a\tsome_file%\xcd\xc0\xa9E(\xb2\xf2\x192 abcd1234abcd1234abcd1234abcd1234'
key=b'fh:fedcba9876543210' value=b'\x00*\x00\x01\x00\x01'
key=b'fh:\xab\xcd\x124\xab\xcd\x124\xab\xcd\x124\xab\xcd\x124' value=b'\x00*\x00\x01\x00\x01'
key=b'r:\x00*' value=b'\x08\x01\x10*\x1a\x10\x8e\xc8t=\xb4\xfcGp\x85C\xd4_\xd5\x13%\xa7" run from jupyter for development*\x05koppa=\x9a\x99\x99?R\x11C:\\some\\start\\dirX\t`\x185\x9a\x99\x99?B\x0binitializedP\x0

In [22]:
the_db.max_run_id()


42

In [53]:
# list runs

rec_type = b'r:'
prefix = rec_type

with env.begin(write=False) as txn, txn.cursor() as cur:
    if cur.set_range(prefix):
        while cur.key().startswith(prefix):
            run_id = int.from_bytes(cur.key()[len(rec_type):], byteorder='big')
            rec = record_pb2.RunRecord()
            rec.ParseFromString(cur.value())
            print(
                f'run {run_id},',
                f'uuid={rec.uuid},',
                f'path={rec.path},',
            )
            cur.next()

run 42, uuid=b'C:\\some\\start\\dir', path=run from jupyter for development,


In [42]:
# list directories from a specific run

rec_type = b'd:'
run_id = Id(42)
prefix = rec_type + run_id.to_bytes()

with env.begin(write=False) as txn, txn.cursor() as cur:
    if cur.set_range(prefix):
        while cur.key().startswith(prefix):
            id_parts = CompositeId.bytes_to_parts(cur.key()[2:])
            assert len(id_parts) == 2
            rec = record_pb2.DirRecord()
            rec.ParseFromString(cur.value())
            print(
                f'dir {CompositeId.bytes_to_parts(cur.key()[len(rec_type):])},',
                f'path={rec.path},',
                f'parent={rec.parent_id},',
            )
            cur.next()

dir (42, 1), path=., parent=b'',
dir (42, 2), path=, parent=b'',


In [57]:
# list files from a specific run

rec_type = b'f:'
run_id = Id(42)
prefix = rec_type + run_id.to_bytes()

with env.begin(write=False) as txn, txn.cursor() as cur:
    if cur.set_range(prefix):
        while cur.key().startswith(prefix):
            id_bytes = cur.key()[len(rec_type):]
            id_parts = CompositeId.bytes_to_parts(id_bytes)
            assert len(id_parts) == 3
            _, dir_id_val, file_id_val = id_parts
            rec = record_pb2.FileRecord()
            rec.ParseFromString(cur.value())
            # assert rec.id == id_bytes
            dir_key = b'd:' + id_bytes[:4]
            dir_val = txn.get(dir_key)
            if not dir_val:
                raise ValueError(f'No value found for key {dir_key}')
            dir_rec = record_pb2.DirRecord()
            dir_rec.ParseFromString(dir_val)
            print(
                f'file {CompositeId.bytes_to_parts(cur.key()[len(rec_type):])},',
                f'name={rec.name},',
                f'location={dir_rec.path},',
            )
            cur.next()

file (42, 1, 1), name=somothername, location=.,
file (42, 1, 2), name=some_file, location=.,


In [20]:
# Restore Run obj from db

rrun42 = the_db.get_run(Id(42))
rrun42.id, rrun42.root_dir


Id(42)

In [56]:
# look for duplicate file hashes

rec_type = b'fh:'

with env.begin(write=False) as txn, txn.cursor() as cur:
    if cur.set_range(prefix):
        last_key, last_value = b'', b''
        files_w_same_hash = []
        while cur.key().startswith(prefix):
            if cur.key() == last_key:
                if not files_w_same_hash:
                    files_w_same_hash.append(last_value)
                files_w_same_hash.append(cur.value())
            elif files_w_same_hash:
                print("Files with same hash:", [CompositeId.bytes_to_parts(f) for f in files_w_same_hash])
                files_w_same_hash = []
            last_key, last_value = cur.key(), cur.value()
            cur.next()

In [8]:
import uuid
u: uuid.UUID = uuid.uuid4()
u2 = uuid.UUID(bytes=u.bytes)
u, u2

(UUID('238098b5-265f-401f-9a60-bdbd9ef7ba29'),
 UUID('238098b5-265f-401f-9a60-bdbd9ef7ba29'))